# Genotype-only CNN external validation: train on `dataset.csv`, test on `unseen_drug.csv`

This notebook keeps `unseen_drug.csv` completely untouched during model selection.

Training is performed in two stages for each seed:
1. **Epoch selection** using an internal drug-group-aware validation split from `dataset.csv`.
2. **Final refit** from scratch on all 576 main samples for the selected number of epochs.
3. **External test** once on `unseen_drug.csv`.

Default seeds: 1001, 2001, 3001. External performance is summarized as mean ± sample SD across seeds.

Required local files/folders:
- `dataset.csv`
- `unseen_drug.csv`
- `onehot_7/`

In [1]:
import os
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt

MAIN_FILE = "dataset.csv"
EXTERNAL_FILE = "unseen_drug.csv"
ONEHOT_FOLDER = "onehot_7"

SEEDS = [1001, 2001, 3001]
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 15
TARGET_VAL_N = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

/data/home/bmi-lab/anaconda3/envs/nayb37/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
main_df = pd.read_csv(MAIN_FILE)
external_df = pd.read_csv(EXTERNAL_FILE)

required_main = {"Variant", "Drug", "Class"}
required_external = {"Variant", "Drug", "Class"}

missing_main = required_main - set(main_df.columns)
missing_external = required_external - set(external_df.columns)
if missing_main:
    raise ValueError(f"Missing columns in {MAIN_FILE}: {missing_main}")
if missing_external:
    raise ValueError(f"Missing columns in {EXTERNAL_FILE}: {missing_external}")

label_encoder = LabelEncoder()
main_df["Class_encoded"] = label_encoder.fit_transform(main_df["Class"])
external_df["Class_encoded"] = label_encoder.transform(external_df["Class"])

class_mapping = {i: x for i, x in enumerate(label_encoder.classes_)}
print("Class mapping:", class_mapping)
print("Main samples:", len(main_df), "| drugs:", main_df["Drug"].nunique())
print("External samples:", len(external_df), "| drugs:", external_df["Drug"].nunique())

main_drugs = set(main_df["Drug"].astype(str))
external_drugs = set(external_df["Drug"].astype(str))
overlap = sorted(main_drugs & external_drugs)

print("External drugs:", sorted(external_drugs))
print("Drug overlap with main dataset:", overlap)
if overlap:
    raise ValueError(
        "External set is intended to be unseen-drug validation, but these drugs "
        f"also occur in the main dataset: {overlap}"
    )

print("\nMain class counts:")
print(main_df["Class"].value_counts())
print("\nExternal class counts:")
print(external_df["Class"].value_counts())

Class mapping: {0: 'de', 1: 'f', 2: 'nor'}
Main samples: 622 | drugs: 59
External samples: 53 | drugs: 5
External drugs: ['amphetamine', 'aripiprazole', 'dapoxetine', 'tamsulosin', 'tedatioxetine']
Drug overlap with main dataset: []

Main class counts:
de     250
f      195
nor    177
Name: Class, dtype: int64

External class counts:
de     28
nor    21
f       4
Name: Class, dtype: int64


In [3]:
TARGET_ROWS = 4330
TARGET_COLS = 7

onehot_matrices = {}
for file in os.listdir(ONEHOT_FOLDER):
    if not file.endswith(".csv"):
        continue
    variant_name = file[:-4].replace("_", ".")
    matrix = pd.read_csv(os.path.join(ONEHOT_FOLDER, file), header=None).values
    if matrix.shape[0] > TARGET_ROWS or matrix.shape[1] > TARGET_COLS:
        raise ValueError(f"{file}: matrix shape {matrix.shape} exceeds target")
    padded = torch.zeros((TARGET_ROWS, TARGET_COLS), dtype=torch.float32)
    padded[:matrix.shape[0], :matrix.shape[1]] = torch.tensor(matrix, dtype=torch.float32)
    onehot_matrices[variant_name] = padded

print("Loaded one-hot matrices:", len(onehot_matrices))

def split_diplotype(variant):
    parts = [x.strip() for x in str(variant).split(";") if x.strip()]
    if len(parts) != 2:
        raise ValueError(f"Expected exactly two haplotypes separated by ';': {variant}")
    return parts

def map_variant_to_onehot(variant):
    hap1, hap2 = split_diplotype(variant)
    missing = [h for h in (hap1, hap2) if h not in onehot_matrices]
    if missing:
        raise KeyError(
            f"Missing one-hot matrix for {missing} in diplotype {variant}. "
            "External validation should not silently replace missing alleles with zeros."
        )
    return torch.vstack([onehot_matrices[hap1], onehot_matrices[hap2]])

def make_cnn_tensor(df):
    return torch.stack([map_variant_to_onehot(v) for v in df["Variant"]]).unsqueeze(1)

main_cnn = make_cnn_tensor(main_df)
external_cnn = make_cnn_tensor(external_df)
main_y = torch.tensor(main_df["Class_encoded"].values, dtype=torch.long)
external_y = torch.tensor(external_df["Class_encoded"].values, dtype=torch.long)

print("Main CNN:", tuple(main_cnn.shape))
print("External CNN:", tuple(external_cnn.shape))

Loaded one-hot matrices: 361
Main CNN: (622, 1, 8660, 7)
External CNN: (53, 1, 8660, 7)


In [4]:
def compute_macro_auc(y_true, y_probs, num_classes=3):
    vals = []
    for i in range(num_classes):
        y_bin = (y_true == i).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            vals.append(np.nan)
        else:
            vals.append(roc_auc_score(y_bin, y_probs[:, i]))
    return float(np.nanmean(vals))

def compute_macro_aupr(y_true, y_probs, num_classes=3):
    vals = []
    for i in range(num_classes):
        y_bin = (y_true == i).astype(int)
        if y_bin.sum() == 0:
            vals.append(np.nan)
        else:
            vals.append(average_precision_score(y_bin, y_probs[:, i]))
    return float(np.nanmean(vals))

def metric_dict(y_true, y_probs):
    y_pred = y_probs.argmax(axis=1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auc": compute_macro_auc(y_true, y_probs, y_probs.shape[1]),
        "aupr": compute_macro_aupr(y_true, y_probs, y_probs.shape[1]),
        "y_pred": y_pred
    }

def choose_internal_split(df, target_val_n=TARGET_VAL_N, random_state=777):
    # Drug-group-aware internal validation. External data are never used here.
    cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=random_state)
    candidates = []
    for tr, va in cv.split(
        np.zeros(len(df)),
        df["Class_encoded"].values,
        groups=df["Drug"].astype(str).values
    ):
        candidates.append((abs(len(va) - target_val_n), tr, va))
    _, tr, va = min(candidates, key=lambda x: x[0])
    assert set(df.iloc[tr]["Drug"]).isdisjoint(set(df.iloc[va]["Drug"]))
    return tr, va

internal_train_idx, internal_val_idx = choose_internal_split(main_df)
print("Internal epoch-selection split:",
      len(internal_train_idx), "train /", len(internal_val_idx), "validation")
print("Train drugs:", main_df.iloc[internal_train_idx]["Drug"].nunique(),
      "| Val drugs:", main_df.iloc[internal_val_idx]["Drug"].nunique())

Internal epoch-selection split: 583 train / 39 validation
Train drugs: 54 | Val drugs: 5


In [5]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=(20, 7), stride=(1, 1))
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 1))
        self.conv2 = nn.Conv2d(8, 16, kernel_size=(20, 1), stride=(1, 1))
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 1))
        self.fc1 = nn.Linear(34400, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [6]:
class GenotypeDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx].float(), self.y[idx].long()

main_dataset = GenotypeDataset(main_cnn, main_y)
external_dataset = GenotypeDataset(external_cnn, external_y)

criterion = nn.CrossEntropyLoss().to(device)

def make_loader(dataset, indices=None, shuffle=False, seed=0):
    ds = dataset if indices is None else torch.utils.data.Subset(dataset, indices)
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=shuffle,
        generator=g if shuffle else None, num_workers=0
    )

def evaluate_cnn(model, loader):
    model.eval()
    ys, logits = [], []
    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += criterion(out, y).item()
            ys.append(y.cpu())
            logits.append(out.cpu())
    y_true = torch.cat(ys).numpy()
    y_probs = F.softmax(torch.cat(logits), dim=1).numpy()
    out = metric_dict(y_true, y_probs)
    out["loss"] = total_loss / len(loader)
    out["y_true"] = y_true
    out["y_probs"] = y_probs
    return out

## Train on main data and evaluate the untouched external set

The internal validation split is used only to select the epoch count. The model is then reinitialized and refit on **all main samples** for that number of epochs before external testing.

In [7]:
# Genotype-only learning rate is kept as an explicit independent setting.
# Change only if you have a pre-specified genotype-only LR from your finalized analysis.
LEARNING_RATE = 1e-3

def select_epoch_cnn(seed):
    set_seed(seed)
    train_loader = make_loader(main_dataset, internal_train_idx, shuffle=True, seed=seed)
    val_loader = make_loader(main_dataset, internal_val_idx, shuffle=False)

    model = SimpleCNN(num_classes=len(label_encoder.classes_)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_f1, best_epoch, wait = -np.inf, 1, 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        val = evaluate_cnn(model, val_loader)
        history.append({
            "epoch": epoch,
            "train_loss": np.mean(losses),
            "val_loss": val["loss"],
            "val_f1": val["f1"]
        })

        if val["f1"] > best_f1:
            best_f1, best_epoch, wait = val["f1"], epoch, 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    return best_epoch, best_f1, pd.DataFrame(history)

def refit_and_test_cnn(seed, epochs):
    set_seed(seed)
    full_loader = make_loader(main_dataset, shuffle=True, seed=seed)
    external_loader = make_loader(external_dataset, shuffle=False)

    model = SimpleCNN(num_classes=len(label_encoder.classes_)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in full_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    ext = evaluate_cnn(model, external_loader)
    return model, ext

results, prediction_rows, histories = [], [], {}

for seed in SEEDS:
    best_epoch, best_val_f1, history = select_epoch_cnn(seed)
    model, ext = refit_and_test_cnn(seed, best_epoch)
    histories[seed] = history

    results.append({
        "seed": seed, "best_epoch": best_epoch, "best_val_f1": best_val_f1,
        "accuracy": ext["accuracy"], "precision": ext["precision"],
        "recall": ext["recall"], "f1": ext["f1"],
        "auc": ext["auc"], "aupr": ext["aupr"]
    })

    for i in range(len(external_df)):
        row = {
            "seed": seed,
            "external_index": i,
            "Variant": external_df.iloc[i]["Variant"],
            "Drug": external_df.iloc[i]["Drug"],
            "Class": external_df.iloc[i]["Class"],
            "y_true": int(ext["y_true"][i]),
            "y_pred": int(ext["y_pred"][i])
        }
        for c in range(ext["y_probs"].shape[1]):
            row[f"prob_{class_mapping[c]}"] = ext["y_probs"][i, c]
        prediction_rows.append(row)

    print(f"Seed {seed} | best epoch={best_epoch} | val F1={best_val_f1:.4f} | "
          f"external Acc={ext['accuracy']:.4f} F1={ext['f1']:.4f} "
          f"AUROC={ext['auc']:.4f} AUPR={ext['aupr']:.4f}")

results_df = pd.DataFrame(results)
predictions_df = pd.DataFrame(prediction_rows)

summary = []
for m in ["accuracy","precision","recall","f1","auc","aupr"]:
    summary.append({
        "metric": m,
        "mean": results_df[m].mean(),
        "sd": results_df[m].std(ddof=1)
    })
summary_df = pd.DataFrame(summary)

display(results_df)
display(summary_df)

results_df.to_csv("external_genotype_only_seed_results.csv", index=False)
summary_df.to_csv("external_genotype_only_summary.csv", index=False)
predictions_df.to_csv("external_genotype_only_predictions.csv", index=False)

Seed 1001 | best epoch=26 | val F1=0.8787 | external Acc=0.7736 F1=0.7647 AUROC=0.9213 AUPR=0.8750
Seed 2001 | best epoch=22 | val F1=0.8787 | external Acc=0.6981 F1=0.7126 AUROC=0.8858 AUPR=0.8447
Seed 3001 | best epoch=27 | val F1=0.8586 | external Acc=0.7547 F1=0.7365 AUROC=0.9119 AUPR=0.8561


,seed,best_epoch,best_val_f1,accuracy,precision,recall,f1,auc,aupr
0,1001,26,0.878676,0.773585,0.764029,0.765873,0.764729,0.921283,0.874972
1,2001,22,0.878676,0.698113,0.722222,0.722222,0.712585,0.885758,0.844703
2,3001,27,0.858586,0.754717,0.772792,0.734127,0.736520,0.911875,0.856126


,metric,mean,sd
0,accuracy,0.742138,0.039277
1,precision,0.753014,0.027024
2,recall,0.740741,0.022564
3,f1,0.737944,0.026101
4,auc,0.906305,0.018405
5,aupr,0.858600,0.015286
